In [1]:
import pandas as pd
import numpy as np

In [2]:
path = "../data/processed/movies_reindexed.parquet"

In [3]:
df = pd.read_parquet(path)

In [4]:
df.head()

,movie_id,title,genres,item_idx
0,1,Toy Story (1995),Animation|Children's|Comedy,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1
2,3,Grumpier Old Men (1995),Comedy|Romance,2
3,4,Waiting to Exhale (1995),Comedy|Drama,3
4,5,Father of the Bride Part II (1995),Comedy,4


In [5]:
df.shape

(3706, 4)

In [6]:
years = df['title'].str.split('(').str[-1].str.rstrip(')').astype('Int64')
years

0       1995
1       1995
2       1995
3       1995
4       1995
        ... 
3701    2000
3702    2000
3703    2000
3704    2000
3705    2000
Name: title, Length: 3706, dtype: Int64

In [7]:
title = df['title'].str.split('(').str[0]
title

0                         Toy Story 
1                           Jumanji 
2                  Grumpier Old Men 
3                 Waiting to Exhale 
4       Father of the Bride Part II 
                    ...             
3701               Meet the Parents 
3702            Requiem for a Dream 
3703                      Tigerland 
3704               Two Family House 
3705                 Contender, The 
Name: title, Length: 3706, dtype: object

In [8]:
df['years'] = years
df['title'] = title
df['genres'] = df['genres'].str.split('|')

In [9]:
df.head()

,movie_id,title,genres,item_idx,years
0,1,Toy Story,"[Animation, Children's, Comedy]",0,1995
1,2,Jumanji,"[Adventure, Children's, Fantasy]",1,1995
2,3,Grumpier Old Men,"[Comedy, Romance]",2,1995
3,4,Waiting to Exhale,"[Comedy, Drama]",3,1995
4,5,Father of the Bride Part II,[Comedy],4,1995


In [10]:
from sklearn.preprocessing import MultiLabelBinarizer

In [11]:
mlb = MultiLabelBinarizer()

In [12]:
genres = mlb.fit_transform(df['genres'])
all_genres = list(mlb.classes_)

In [13]:
genres.shape

(3706, 18)

In [14]:
all_genres

['Action',
 'Adventure',
 'Animation',
 "Children's",
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western']

In [15]:
genres[0]

array([0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [17]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20000
)

In [18]:
X_tfidf = vectorizer.fit_transform(df['title'])

In [19]:
X_tfidf.shape

(3706, 3658)

In [20]:
from sklearn.decomposition import TruncatedSVD

In [21]:
svd = TruncatedSVD(n_components=128, random_state=42)

In [22]:
title_svd = svd.fit_transform(X_tfidf)

In [23]:
title_svd.shape

(3706, 128)

In [24]:
from sklearn.preprocessing import MinMaxScaler

In [25]:
scaler = MinMaxScaler()

In [26]:
y = df[['years']].to_numpy()

In [27]:
scaled_y = scaler.fit_transform(y)

In [28]:
scaled_y

array([[0.9382716],
       [0.9382716],
       [0.9382716],
       ...,
       [1.       ],
       [1.       ],
       [1.       ]])

In [29]:
y.shape

(3706, 1)

In [30]:
from pathlib import Path

In [31]:
OUT_DIR = Path("../data/processed/item_features")
OUT_DIR.mkdir(parents=False, exist_ok=True)

np.save(OUT_DIR / "title_svd128.npy", title_svd.astype(np.float32))
np.save(OUT_DIR / "genres.npy", genres.astype(np.float32))
np.save(OUT_DIR / "year_minmax.npy", scaled_y.astype(np.float32))


In [32]:
df[["movie_id", "item_idx", "title", "years", "genres"]].to_parquet(
    OUT_DIR / "item_meta.parquet", index=False
)